# Notebook 4 — Trust & Uncertainty Estimation

## Explainable Apple Plant Disease Detection

### Objective

This notebook builds the **Trust & Uncertainty layer** of the project.

The previous notebooks established:

1. A clean four-class Apple dataset.
2. Transfer-learning model comparison.
3. ResNet50 as the selected classifier.
4. Grad-CAM explanations for the complete test set.

Now we move from:

> **What did the model predict, and where did it look?**

to:

> **How reliable and certain is that prediction?**

This notebook therefore measures prediction confidence, predictive uncertainty, calibration, and a combined trust/reliability score that can later be used by the RAG, LLM, human-in-the-loop, and web-dashboard components.

## Project Position

```text
Notebook 1
Data Preparation
        ↓
Notebook 2
Transfer Learning + Model Selection
        ↓
Notebook 3
Grad-CAM Explainability
        ↓
Notebook 4
Trust & Uncertainty Estimation  ← CURRENT
        ↓
Notebook 5
RAG Knowledge Retrieval
        ↓
Notebook 6
LLM Explanation
        ↓
Notebook 7
Human-in-the-Loop + Feedback
        ↓
Web Dashboard
```

In [ ]:
import os
import json
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from google.colab import drive

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/XAI_Project"
DATA_DIR = os.path.join(PROJECT_DIR, "processed")
MODEL_DIR = os.path.join(PROJECT_DIR, "saved_models")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
TRUST_DIR = os.path.join(RESULTS_DIR, "trust_uncertainty")

os.makedirs(TRUST_DIR, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Trust/uncertainty output directory:", TRUST_DIR)

In [ ]:
# Reproducibility and device

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 4
IMAGE_SIZE = 224
BATCH_SIZE = 64

CLASS_NAMES = [
    "Apple Healthy",
    "Apple Scab",
    "Apple Black Rot",
    "Apple Cedar Apple Rust"
]

CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

print("Device:", DEVICE)
print("Number of classes:", NUM_CLASSES)
print("Input image size:", f"3 x {IMAGE_SIZE} x {IMAGE_SIZE}")
print("Batch size:", BATCH_SIZE)
print("Classes:", CLASS_NAMES)

## 1. Inputs

The notebook uses:

- The saved four-class ResNet50 model.
- The processed validation and test tensors.
- The Grad-CAM prediction/statistics report produced by Notebook 3, when available.

### Image input dimension

```text
3 × 224 × 224
```

### Model output dimension

```text
4 logits
```

corresponding to:

```text
0 → Apple Healthy
1 → Apple Scab
2 → Apple Black Rot
3 → Apple Cedar Apple Rust
```

The four logits are converted to four probabilities using Softmax.

In [ ]:
def load_processed_split(filename):
    path = os.path.join(DATA_DIR, filename)
    obj = torch.load(path, map_location="cpu")

    if isinstance(obj, (tuple, list)) and len(obj) == 2:
        images, labels = obj
    elif isinstance(obj, dict):
        possible_x = ["images", "X", "data", "inputs"]
        possible_y = ["labels", "y", "targets"]

        x_key = next((k for k in possible_x if k in obj), None)
        y_key = next((k for k in possible_y if k in obj), None)

        if x_key is None or y_key is None:
            raise ValueError(f"Could not identify image/label keys in {filename}. Keys: {list(obj.keys())}")

        images, labels = obj[x_key], obj[y_key]
    else:
        raise ValueError(f"Unsupported format in {filename}: {type(obj)}")

    images = torch.as_tensor(images)
    labels = torch.as_tensor(labels).long()

    return images, labels

val_images, val_labels = load_processed_split("val.pt")
test_images, test_labels = load_processed_split("test.pt")

print("Validation images:", val_images.shape)
print("Validation labels:", val_labels.shape)
print("Test images:", test_images.shape)
print("Test labels:", test_labels.shape)

In [ ]:
# Create DataLoaders

val_loader = DataLoader(
    TensorDataset(val_images, val_labels),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(test_images, test_labels),
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

## 2. Load the Trained ResNet50

The same architecture used in the transfer-learning and Grad-CAM notebooks is reconstructed.

The final fully connected layer is changed to four outputs for the Apple task.

The notebook first looks for:

```text
best_overall_model.pt
```

and falls back to:

```text
resnet50_best.pt
```

if necessary.

In [ ]:
def find_checkpoint():
    candidates = [
        os.path.join(MODEL_DIR, "best_overall_model.pt"),
        os.path.join(MODEL_DIR, "resnet50_best.pt"),
        os.path.join(MODEL_DIR, "best_resnet50.pt"),
    ]

    for path in candidates:
        if os.path.exists(path):
            return path

    raise FileNotFoundError(
        "No ResNet50 checkpoint found. Checked:\n" + "\n".join(candidates)
    )

CHECKPOINT_PATH = find_checkpoint()
print("Using checkpoint:", CHECKPOINT_PATH)

In [ ]:
def build_resnet50():
    try:
        model = models.resnet50(weights=None)
    except TypeError:
        model = models.resnet50(pretrained=False)

    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model

model = build_resnet50()

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

if isinstance(checkpoint, dict):
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    elif "model" in checkpoint and isinstance(checkpoint["model"], dict):
        state_dict = checkpoint["model"]
    else:
        # Some projects save the raw state_dict directly.
        state_dict = checkpoint
else:
    state_dict = checkpoint

# Remove a possible DataParallel prefix.
clean_state_dict = {}
for key, value in state_dict.items():
    clean_key = key.replace("module.", "", 1) if key.startswith("module.") else key
    clean_state_dict[clean_key] = value

missing, unexpected = model.load_state_dict(clean_state_dict, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

model = model.to(DEVICE)
model.eval()

print("ResNet50 loaded successfully.")

## 3. Prediction Probabilities

For each image, the model produces four logits.

Softmax converts them into probabilities:

```text
p_i = exp(z_i) / Σ exp(z_j)
```

The predicted class is the class with the largest probability.

The maximum probability is the model's **confidence**.

Confidence is useful, but it is **not the same thing as calibrated reliability**. A neural network can be very confident and still be wrong. That is why this notebook also measures uncertainty and calibration.

In [ ]:
@torch.no_grad()
def collect_predictions(model, loader):
    all_probs = []
    all_logits = []
    all_labels = []

    for images, labels in loader:
        images = images.to(DEVICE)

        logits = model(images)
        probs = torch.softmax(logits, dim=1)

        all_logits.append(logits.cpu())
        all_probs.append(probs.cpu())
        all_labels.append(labels.cpu())

    logits = torch.cat(all_logits).numpy()
    probs = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()

    preds = np.argmax(probs, axis=1)
    confidence = np.max(probs, axis=1)

    return logits, probs, labels, preds, confidence

val_logits, val_probs, val_y, val_preds, val_conf = collect_predictions(model, val_loader)
test_logits, test_probs, test_y, test_preds, test_conf = collect_predictions(model, test_loader)

print("Validation predictions:", len(val_y))
print("Test predictions:", len(test_y))
print("Probability shape:", test_probs.shape)
print("Logit shape:", test_logits.shape)

In [ ]:
# Basic test-set classification metrics

test_accuracy = accuracy_score(test_y, test_preds)
test_precision = precision_score(test_y, test_preds, average="macro", zero_division=0)
test_recall = recall_score(test_y, test_preds, average="macro", zero_division=0)
test_f1 = f1_score(test_y, test_preds, average="macro", zero_division=0)

print("Test Accuracy :", f"{test_accuracy:.4f}")
print("Test Precision:", f"{test_precision:.4f}")
print("Test Recall   :", f"{test_recall:.4f}")
print("Test Macro F1 :", f"{test_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        test_y,
        test_preds,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )
)

## 4. Predictive Uncertainty

A simple uncertainty measure for a multi-class probability distribution is **predictive entropy**:

```text
H(p) = - Σ p_i log(p_i)
```

For four classes, the maximum entropy is:

```text
log(4)
```

We normalize entropy to the range 0–1:

```text
Normalized Entropy = H(p) / log(4)
```

Interpretation:

- Near 0 → probability distribution is concentrated on one class.
- Near 1 → probability distribution is spread across the four classes.

This is a probability-distribution uncertainty measure. It should not be interpreted as a perfect measure of epistemic uncertainty.

In [ ]:
def normalized_entropy(probabilities):
    probabilities = np.clip(probabilities, 1e-12, 1.0)
    entropy = -np.sum(probabilities * np.log(probabilities), axis=1)
    return entropy / np.log(probabilities.shape[1])

val_entropy = normalized_entropy(val_probs)
test_entropy = normalized_entropy(test_probs)

print("Validation entropy range:", (float(val_entropy.min()), float(val_entropy.max())))
print("Test entropy range:", (float(test_entropy.min()), float(test_entropy.max())))

## 5. Margin-Based Uncertainty

Another useful signal is the difference between the largest and second-largest predicted probabilities.

```text
Margin = p_top1 - p_top2
```

A small margin means the model has similar probabilities for competing classes.

We convert it into an uncertainty value:

```text
Margin Uncertainty = 1 - Margin
```

This is another descriptive uncertainty signal and is used together with entropy rather than treated as a standalone truth measure.

In [ ]:
def margin_uncertainty(probabilities):
    sorted_probs = np.sort(probabilities, axis=1)
    top1 = sorted_probs[:, -1]
    top2 = sorted_probs[:, -2]
    margin = top1 - top2
    uncertainty = 1.0 - margin
    return margin, uncertainty

val_margin, val_margin_unc = margin_uncertainty(val_probs)
test_margin, test_margin_unc = margin_uncertainty(test_probs)

print("Test margin range:", (float(test_margin.min()), float(test_margin.max())))
print("Test margin uncertainty range:", (float(test_margin_unc.min()), float(test_margin_unc.max())))

## 6. Calibration

Calibration asks:

> When the model says it is 90% confident, is it correct approximately 90% of the time?

This is different from raw accuracy.

We use **Expected Calibration Error (ECE)** to summarize the difference between confidence and empirical accuracy across confidence bins.

ECE is calculated on the **validation set** and then reported on the test set.

The validation set is important because calibration information should not be tuned directly on the final test set.

In [ ]:
def expected_calibration_error(y_true, y_pred, confidence, n_bins=10):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    confidence = np.asarray(confidence)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    rows = []

    n = len(y_true)

    for i in range(n_bins):
        left = bin_edges[i]
        right = bin_edges[i + 1]

        if i == n_bins - 1:
            mask = (confidence >= left) & (confidence <= right)
        else:
            mask = (confidence >= left) & (confidence < right)

        count = int(mask.sum())

        if count == 0:
            rows.append({
                "bin": i,
                "lower": left,
                "upper": right,
                "count": 0,
                "mean_confidence": np.nan,
                "accuracy": np.nan,
                "gap": np.nan
            })
            continue

        bin_conf = confidence[mask].mean()
        bin_acc = (y_pred[mask] == y_true[mask]).mean()
        gap = abs(bin_acc - bin_conf)

        ece += (count / n) * gap

        rows.append({
            "bin": i,
            "lower": left,
            "upper": right,
            "count": count,
            "mean_confidence": bin_conf,
            "accuracy": bin_acc,
            "gap": gap
        })

    return float(ece), pd.DataFrame(rows)

val_ece, val_calibration_bins = expected_calibration_error(
    val_y, val_preds, val_conf, n_bins=10
)

test_ece, test_calibration_bins = expected_calibration_error(
    test_y, test_preds, test_conf, n_bins=10
)

print("Validation ECE:", f"{val_ece:.6f}")
print("Test ECE:", f"{test_ece:.6f}")

In [ ]:
# Reliability diagram

plt.figure(figsize=(7, 6))

valid_bins = val_calibration_bins.dropna(subset=["mean_confidence", "accuracy"])

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)

plt.plot(
    valid_bins["mean_confidence"],
    valid_bins["accuracy"],
    marker="o",
    label="Validation model"
)

plt.xlabel("Mean confidence")
plt.ylabel("Empirical accuracy")
plt.title("Validation Reliability Diagram")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

reliability_path = os.path.join(TRUST_DIR, "validation_reliability_diagram.png")
plt.savefig(reliability_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", reliability_path)

## 7. Confidence vs Correctness

A central purpose of the trust layer is to determine whether confidence is associated with actual correctness.

We therefore compare:

- Correct predictions
- Incorrect predictions

across confidence ranges.

This does not prove causation. It simply evaluates how useful the confidence signal is for identifying reliable versus unreliable predictions.

In [ ]:
test_correct = (test_preds == test_y)

confidence_summary = pd.DataFrame({
    "correct": test_correct,
    "confidence": test_conf,
    "entropy": test_entropy,
    "margin": test_margin,
    "margin_uncertainty": test_margin_unc
})

print("Correct predictions:", int(test_correct.sum()))
print("Incorrect predictions:", int((~test_correct).sum()))

print("\nMean confidence:")
print(confidence_summary.groupby("correct")["confidence"].mean())

print("\nMean normalized entropy:")
print(confidence_summary.groupby("correct")["entropy"].mean())

In [ ]:
# Confidence distributions

plt.figure(figsize=(8, 5))

plt.hist(
    test_conf[test_correct],
    bins=20,
    alpha=0.7,
    label="Correct predictions"
)

if (~test_correct).any():
    plt.hist(
        test_conf[~test_correct],
        bins=20,
        alpha=0.7,
        label="Incorrect predictions"
    )

plt.xlabel("Prediction confidence")
plt.ylabel("Number of images")
plt.title("Confidence Distribution by Prediction Correctness")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

confidence_dist_path = os.path.join(
    TRUST_DIR,
    "confidence_distribution_correct_vs_incorrect.png"
)
plt.savefig(confidence_dist_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", confidence_dist_path)

## 8. Incorporating Grad-CAM Information

Notebook 3 generated Grad-CAM statistics for the test set.

Grad-CAM statistics are used here only as **descriptive explanation features**.

They are not treated as ground-truth localization quality because the dataset does not provide:

- Disease bounding boxes
- Pixel-level segmentation masks
- Ground-truth disease localization

Therefore:

> **No IoU or bounding-box score is used in this project.**

We will attempt to load the Grad-CAM statistics report. If it is unavailable or does not match the test-set size, the trust model will continue using validated predictive signals without inventing missing Grad-CAM measurements.

In [ ]:
def find_gradcam_csv():
    candidates = [
        os.path.join(RESULTS_DIR, "gradcam", "gradcam_statistics.csv"),
        os.path.join(RESULTS_DIR, "gradcam_prediction_report.csv"),
        os.path.join(RESULTS_DIR, "gradcam", "gradcam_all_test_statistics.csv"),
    ]

    for path in candidates:
        if os.path.exists(path):
            return path

    return None

GRADCAM_CSV = find_gradcam_csv()

if GRADCAM_CSV is None:
    print("No Grad-CAM CSV found.")
    gradcam_df = None
else:
    print("Loading Grad-CAM report:", GRADCAM_CSV)
    gradcam_df = pd.read_csv(GRADCAM_CSV)
    print("Grad-CAM report shape:", gradcam_df.shape)
    print("Columns:", list(gradcam_df.columns))

In [ ]:
# Identify useful numeric Grad-CAM columns when available.

gradcam_numeric_candidates = []

if gradcam_df is not None:
    preferred_names = [
        "max_activation",
        "mean_activation",
        "high_activation_area_percent",
        "high_activation_percentage",
        "high_activation_area",
    ]

    for name in preferred_names:
        if name in gradcam_df.columns:
            gradcam_numeric_candidates.append(name)

print("Grad-CAM numeric signals found:", gradcam_numeric_candidates)

## 9. Trust Score Design

The trust score is designed as a **reliability indicator**, not as a mathematical guarantee that a prediction is correct.

The most defensible core signals available without localization ground truth are:

1. Calibrated confidence.
2. Low predictive entropy.
3. Large top-1/top-2 probability margin.

We first transform uncertainty into reliability:

```text
Entropy Reliability = 1 - normalized entropy
Margin Reliability = 1 - margin uncertainty
```

A simple transparent fusion is:

```text
Core Trust =
    0.50 × Calibrated Confidence
  + 0.25 × Entropy Reliability
  + 0.25 × Margin Reliability
```

The weights are explicitly defined rather than hidden.

However, because the model can be highly confident while wrong, we also calculate an empirical **confidence reliability mapping** on the validation set. This mapping estimates the observed accuracy associated with different confidence ranges.

The final trust score uses this validation-derived reliability estimate as the primary calibrated signal, while entropy and margin provide additional uncertainty information.

This avoids treating raw confidence as perfectly trustworthy.

In [ ]:
# Validation-derived confidence reliability mapping.
# We use quantile-free fixed confidence bins for reproducibility.

N_CALIB_BINS = 10

def fit_confidence_reliability_map(y_true, y_pred, confidence, n_bins=10):
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []

    for i in range(n_bins):
        left = edges[i]
        right = edges[i + 1]

        if i == n_bins - 1:
            mask = (confidence >= left) & (confidence <= right)
        else:
            mask = (confidence >= left) & (confidence < right)

        count = int(mask.sum())

        if count > 0:
            accuracy = float((y_pred[mask] == y_true[mask]).mean())
            mean_conf = float(confidence[mask].mean())
        else:
            accuracy = np.nan
            mean_conf = np.nan

        rows.append({
            "bin": i,
            "lower": left,
            "upper": right,
            "count": count,
            "mean_confidence": mean_conf,
            "empirical_accuracy": accuracy
        })

    return pd.DataFrame(rows)

calibration_map = fit_confidence_reliability_map(
    val_y,
    val_preds,
    val_conf,
    n_bins=N_CALIB_BINS
)

calibration_map

In [ ]:
def validation_accuracy_for_confidence(confidence_values, calibration_table):
    confidence_values = np.asarray(confidence_values)
    result = np.zeros_like(confidence_values, dtype=float)

    for i, conf in enumerate(confidence_values):
        # Find the corresponding confidence bin.
        if conf >= 1.0:
            bin_idx = len(calibration_table) - 1
        else:
            bin_idx = min(int(conf * N_CALIB_BINS), N_CALIB_BINS - 1)

        row = calibration_table.iloc[bin_idx]
        empirical_accuracy = row["empirical_accuracy"]

        if pd.isna(empirical_accuracy):
            # Fall back to the confidence itself if the validation bin is empty.
            empirical_accuracy = conf

        result[i] = empirical_accuracy

    return np.clip(result, 0.0, 1.0)

test_calibrated_confidence = validation_accuracy_for_confidence(
    test_conf,
    calibration_map
)

val_calibrated_confidence = validation_accuracy_for_confidence(
    val_conf,
    calibration_map
)

print("Calibrated confidence range:",
      float(test_calibrated_confidence.min()),
      float(test_calibrated_confidence.max()))

In [ ]:
# Core trust fusion.

test_entropy_reliability = 1.0 - test_entropy
test_margin_reliability = test_margin

test_trust_score = (
    0.50 * test_calibrated_confidence
    + 0.25 * test_entropy_reliability
    + 0.25 * test_margin_reliability
)

test_trust_score = np.clip(test_trust_score, 0.0, 1.0)

val_entropy_reliability = 1.0 - val_entropy
val_margin_reliability = val_margin

val_trust_score = (
    0.50 * val_calibrated_confidence
    + 0.25 * val_entropy_reliability
    + 0.25 * val_margin_reliability
)

val_trust_score = np.clip(val_trust_score, 0.0, 1.0)

print("Test trust score range:",
      float(test_trust_score.min()),
      float(test_trust_score.max()))

## 10. Trust Levels

For dashboard presentation, the continuous trust score can be converted into descriptive levels.

```text
Trust ≥ 0.80  → High
0.60–0.79     → Medium
< 0.60        → Low
```

These thresholds are **presentation thresholds**, not claims of statistical truth.

The continuous score should remain available to downstream systems. The categorical label is only a human-readable interpretation.

In [ ]:
def trust_level(score):
    if score >= 0.80:
        return "High"
    elif score >= 0.60:
        return "Medium"
    return "Low"

test_trust_levels = np.array([trust_level(x) for x in test_trust_score])

print(pd.Series(test_trust_levels).value_counts())

## 11. Does Trust Relate to Correctness?

We evaluate the trust score against the known test labels.

This is an evaluation step, not a training step.

A useful trust score should generally assign higher reliability to groups with more correct predictions.

We report:

- Mean trust score for correct predictions.
- Mean trust score for incorrect predictions.
- Accuracy at different trust thresholds.

This helps determine whether the trust layer is useful before it is connected to RAG or human review.

In [ ]:
trust_eval = pd.DataFrame({
    "true_label": test_y,
    "predicted_label": test_preds,
    "correct": test_correct,
    "confidence": test_conf,
    "calibrated_confidence": test_calibrated_confidence,
    "entropy": test_entropy,
    "entropy_reliability": test_entropy_reliability,
    "margin": test_margin,
    "margin_reliability": test_margin_reliability,
    "trust_score": test_trust_score,
    "trust_level": test_trust_levels
})

print("Mean trust — correct:",
      trust_eval.loc[trust_eval["correct"], "trust_score"].mean())

if (~test_correct).any():
    print("Mean trust — incorrect:",
          trust_eval.loc[~trust_eval["correct"], "trust_score"].mean())
else:
    print("There are no incorrect test predictions in this evaluation set.")

In [ ]:
threshold_rows = []

for threshold in [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]:
    selected = test_trust_score >= threshold
    count = int(selected.sum())

    if count > 0:
        selected_accuracy = float(test_correct[selected].mean())
    else:
        selected_accuracy = np.nan

    coverage = float(selected.mean())

    threshold_rows.append({
        "trust_threshold": threshold,
        "samples_kept": count,
        "coverage": coverage,
        "accuracy_among_selected": selected_accuracy
    })

trust_threshold_analysis = pd.DataFrame(threshold_rows)
trust_threshold_analysis

In [ ]:
# Trust score distribution by correctness

plt.figure(figsize=(8, 5))

plt.hist(
    test_trust_score[test_correct],
    bins=20,
    alpha=0.7,
    label="Correct predictions"
)

if (~test_correct).any():
    plt.hist(
        test_trust_score[~test_correct],
        bins=20,
        alpha=0.7,
        label="Incorrect predictions"
    )

plt.xlabel("Trust score")
plt.ylabel("Number of images")
plt.title("Trust Score Distribution by Correctness")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

trust_dist_path = os.path.join(
    TRUST_DIR,
    "trust_score_distribution_correct_vs_incorrect.png"
)
plt.savefig(trust_dist_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", trust_dist_path)

## 12. Confusion Matrix

The confusion matrix is retained as a reference because trust should be interpreted alongside the underlying classification performance.

It shows which Apple disease classes are confused with each other.

In [ ]:
cm = confusion_matrix(test_y, test_preds)

plt.figure(figsize=(7, 6))
plt.imshow(cm, interpolation="nearest")
plt.title("Test Set Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=35, ha="right")
plt.yticks(range(NUM_CLASSES), CLASS_NAMES)

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()

cm_path = os.path.join(TRUST_DIR, "test_confusion_matrix.png")
plt.savefig(cm_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", cm_path)

## 13. Final Test Report

The following table is the main output of Notebook 4.

Each row corresponds to one test image and contains:

- Image index
- True class
- Predicted class
- Correctness
- Four class probabilities
- Confidence
- Normalized entropy
- Margin
- Calibrated confidence
- Trust score
- Trust level

This report is designed to be reusable by the future dashboard and by later RAG/LLM logic.

In [ ]:
report = pd.DataFrame({
    "image_index": np.arange(len(test_y)),
    "true_label": test_y,
    "true_class": [CLASS_NAMES[i] for i in test_y],
    "predicted_label": test_preds,
    "predicted_class": [CLASS_NAMES[i] for i in test_preds],
    "correct": test_correct,
    "confidence": test_conf,
    "entropy": test_entropy,
    "entropy_reliability": test_entropy_reliability,
    "margin": test_margin,
    "margin_reliability": test_margin_reliability,
    "calibrated_confidence": test_calibrated_confidence,
    "trust_score": test_trust_score,
    "trust_level": test_trust_levels
})

for class_id, class_name in enumerate(CLASS_NAMES):
    report[f"prob_{class_id}_{class_name.replace(' ', '_')}"] = test_probs[:, class_id]

# If Notebook 3 statistics can be aligned by row order, append them.
if gradcam_df is not None and len(gradcam_df) == len(report):
    for col in gradcam_numeric_candidates:
        report[f"gradcam_{col}"] = pd.to_numeric(
            gradcam_df[col],
            errors="coerce"
        ).values
    print("Grad-CAM statistics aligned and appended to report.")
else:
    print("Grad-CAM statistics were not appended automatically because row counts did not match.")

report_path = os.path.join(TRUST_DIR, "trust_uncertainty_test_report.csv")
report.to_csv(report_path, index=False)

print("Saved:", report_path)
print("Rows:", len(report))
report.head()

In [ ]:
# Save calibration-bin analysis.

calibration_bins_path = os.path.join(
    TRUST_DIR,
    "validation_calibration_bins.csv"
)
calibration_map.to_csv(calibration_bins_path, index=False)

test_calibration_bins_path = os.path.join(
    TRUST_DIR,
    "test_calibration_bins.csv"
)
test_calibration_bins.to_csv(test_calibration_bins_path, index=False)

trust_threshold_path = os.path.join(
    TRUST_DIR,
    "trust_threshold_analysis.csv"
)
trust_threshold_analysis.to_csv(trust_threshold_path, index=False)

print("Saved calibration and threshold reports.")

## 14. Save Configuration for Future Dashboard Use

The dashboard will need the same class mapping, preprocessing information, uncertainty definitions, calibration mapping, and trust-score formula.

The configuration file therefore records the decisions made in this notebook.

This is important because the web application should not silently use a different preprocessing or trust formula from the research notebooks.

In [ ]:
config = {
    "project": "Explainable Apple Plant Disease Detection",
    "model": "ResNet50",
    "checkpoint": CHECKPOINT_PATH,
    "num_classes": NUM_CLASSES,
    "class_mapping": {str(i): CLASS_NAMES[i] for i in range(NUM_CLASSES)},
    "input_shape_chw": [3, IMAGE_SIZE, IMAGE_SIZE],
    "normalization": {
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225]
    },
    "uncertainty": {
        "predictive_entropy": "H(p) / log(num_classes)",
        "margin": "top1_probability - top2_probability",
        "margin_uncertainty": "1 - margin"
    },
    "calibration": {
        "method": "validation confidence-bin empirical accuracy",
        "number_of_bins": N_CALIB_BINS,
        "validation_ece": val_ece,
        "test_ece": test_ece
    },
    "trust_score": {
        "formula": (
            "0.50 * calibrated_confidence + "
            "0.25 * (1 - normalized_entropy) + "
            "0.25 * margin"
        ),
        "weights": {
            "calibrated_confidence": 0.50,
            "entropy_reliability": 0.25,
            "margin_reliability": 0.25
        },
        "levels": {
            "high": ">= 0.80",
            "medium": ">= 0.60 and < 0.80",
            "low": "< 0.60"
        }
    },
    "gradcam": {
        "used_as_ground_truth": False,
        "iou_used": False,
        "bounding_boxes_used": False,
        "note": "Grad-CAM is treated as an explanation/descriptive signal, not localization ground truth."
    }
}

config_path = os.path.join(TRUST_DIR, "trust_uncertainty_config.json")

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print("Saved:", config_path)

# Notebook 4 — Final Summary

## 1. What We Have Done

This notebook implemented the **Trust & Uncertainty Estimation layer** of the Explainable Apple Plant Disease Detection project.

The trained ResNet50 model from the previous notebook was loaded and evaluated on the validation and test datasets.

For every test image, the model produces four class probabilities corresponding to:

```text
0 → Apple Healthy
1 → Apple Scab
2 → Apple Black Rot
3 → Apple Cedar Apple Rust
```

We then calculated:

- Prediction
- Class probabilities
- Prediction confidence
- Predictive entropy
- Entropy-based reliability
- Top-1 vs top-2 probability margin
- Margin-based uncertainty
- Calibration error
- Validation-based calibrated confidence
- Combined trust score
- Trust level

---

## 2. Input

The model receives an RGB Apple leaf image with the following tensor dimension:

```text
3 × 224 × 224
```

For a batch:

```text
Batch Size × 3 × 224 × 224
```

The same ImageNet normalization used in the previous notebooks is maintained:

```text
Mean = [0.485, 0.456, 0.406]
Std  = [0.229, 0.224, 0.225]
```

---

## 3. Model Output

ResNet50 produces:

```text
4 logits
```

which are converted into:

```text
4 class probabilities
```

using Softmax.

The predicted class is the class with the highest probability.

---

## 4. What We Are Trying to Achieve

The goal of this notebook is to distinguish between:

> **Prediction confidence**

and

> **Prediction reliability.**

A model may output a high confidence value even when it is wrong.

Therefore, we analyze confidence together with probability-distribution uncertainty and calibration.

The final Trust Score is a transparent fusion of:

```text
Calibrated Confidence
        +
Low Predictive Entropy
        +
Large Prediction Margin
        ↓
Trust / Reliability Score
```

The score is intended as a decision-support signal, not as a guarantee of correctness.

---

## 5. Trust Score

The current transparent formulation is:

```text
Trust Score =
    0.50 × Calibrated Confidence
  + 0.25 × Entropy Reliability
  + 0.25 × Margin Reliability
```

where:

```text
Entropy Reliability = 1 − Normalized Entropy

Margin Reliability =
    Top-1 Probability − Top-2 Probability
```

The confidence component is mapped using empirical validation-set accuracy in confidence bins rather than blindly assuming that raw confidence is perfectly calibrated.

The trust levels used for presentation are:

```text
Trust ≥ 0.80  → High
0.60–0.79     → Medium
< 0.60        → Low
```

These thresholds are presentation categories and should be validated further as the system evolves.

---

## 6. Calibration

Expected Calibration Error (ECE) was calculated to measure the difference between:

```text
Predicted confidence
```

and:

```text
Observed accuracy
```

across confidence bins.

This is important because a reliable confidence score should correspond reasonably well to actual correctness.

The validation set is used for the confidence-to-reliability mapping, while the test set is retained for final evaluation.

---

## 7. Grad-CAM Relationship

Notebook 3 already provides the visual explanation layer.

Notebook 4 does **not** use IoU or bounding-box accuracy.

The reason is that the Apple dataset contains image-level disease labels but does not provide pixel-level disease masks or disease bounding boxes.

Therefore:

```text
No ground-truth disease localization
        ↓
No meaningful IoU evaluation
        ↓
Grad-CAM remains an explanation signal
```

Grad-CAM-derived statistics may be stored alongside the trust report when they can be safely aligned, but they are not treated as proof that a prediction is correct.

---

## 8. Outputs Saved to Google Drive

The outputs of this notebook are stored under:

```text
XAI_Project/
└── results/
    └── trust_uncertainty/
```

Important outputs include:

```text
trust_uncertainty_test_report.csv
validation_calibration_bins.csv
test_calibration_bins.csv
trust_threshold_analysis.csv
trust_uncertainty_config.json
validation_reliability_diagram.png
confidence_distribution_correct_vs_incorrect.png
trust_score_distribution_correct_vs_incorrect.png
test_confusion_matrix.png
```

The main CSV contains one row for every test image.

---

## 9. How This Helps the Overall Project

Before Notebook 4, our system could provide:

```text
Disease Prediction
+
Confidence
+
Grad-CAM Explanation
```

After Notebook 4, the system can additionally provide:

```text
Disease Prediction
+
Confidence
+
Uncertainty
+
Calibration Information
+
Trust Score
+
Trust Level
+
Grad-CAM Explanation
```

This creates the decision layer required before the knowledge and language components.

---

## 10. How This Connects to RAG

The next stage is **Retrieval-Augmented Generation (RAG)**.

The future pipeline will be:

```text
Apple Leaf
    ↓
ResNet50
    ↓
Prediction
    +
Confidence
    +
Uncertainty
    +
Trust Score
    +
Grad-CAM
    ↓
RAG
    ↓
Relevant Plant-Disease Knowledge
```

For example, a low-trust prediction can trigger a more cautious explanation and additional supporting information.

---

## 11. How This Connects to the LLM

After RAG retrieves relevant disease information, the LLM can combine:

```text
Model Prediction
+
Confidence
+
Trust
+
Grad-CAM Explanation
+
Retrieved Knowledge
```

to produce a human-readable explanation.

The LLM should use the retrieved evidence rather than being treated as the primary disease classifier.

---

## 12. How This Connects to Human-in-the-Loop

Trust becomes particularly important when the model is uncertain.

Conceptually:

```text
High Trust
    ↓
Present prediction + explanation

Low / Uncertain Trust
    ↓
Request additional review
    ↓
Human feedback
```

Human feedback can later be stored as part of the project's feedback/memory mechanism.

---

## 13. How This Helps the Web Dashboard

The future web application can display:

```text
Prediction
Confidence
Uncertainty
Trust Score
Trust Level
Grad-CAM Overlay
Retrieved Disease Information
LLM Explanation
Human Feedback Option
```

For example:

```text
Prediction: Apple Scab
Confidence: 94.2%
Trust Score: 0.87
Trust Level: High

[Original Image] [Grad-CAM Overlay]

Explanation:
...
```

For a low-trust case, the interface can instead emphasize uncertainty and request human review.

---

## 14. Limitations

This notebook has several important limitations:

1. Predictive entropy from a single Softmax output is not a complete measure of all forms of uncertainty.
2. Confidence can remain overconfident even when a model is wrong.
3. The current trust formula is a transparent research baseline and should be validated and refined using additional validation data and deployment observations.
4. Grad-CAM is an explanation method, not ground-truth disease localization.
5. No IoU or bounding-box score is used because the dataset has no corresponding localization annotations.
6. Trust should be interpreted as a decision-support signal, not a guarantee of correctness.
7. The PlantVillage-style dataset may not fully represent real-world field conditions, so final deployment claims should be made cautiously.

---

## 15. Final Architecture After Notebook 4

The project has now reached:

```text
PlantVillage Apple Dataset
            ↓
Data Preparation
            ↓
Four-Class Apple Dataset
            ↓
Transfer Learning
            ↓
ResNet50
            ↓
Prediction + Confidence
            ↓
Grad-CAM Explainability
            ↓
Trust & Uncertainty
            ↓
        NEXT
            ↓
RAG Knowledge Retrieval
            ↓
LLM Explanation
            ↓
Human-in-the-Loop
            ↓
Feedback / Memory
            ↓
Web Dashboard
```

### Final Goal

The objective of the complete project is not simply to build a classifier.

The goal is to build an **Explainable and Trust-Aware Apple Plant Disease Detection System** that can provide:

```text
WHAT?
→ Predicted disease

HOW CONFIDENT?
→ Prediction confidence

HOW UNCERTAIN?
→ Uncertainty measures

WHY?
→ Grad-CAM explanation

HOW RELIABLE?
→ Trust / reliability assessment

WHAT SUPPORTING KNOWLEDGE?
→ RAG

HOW TO EXPLAIN IT?
→ LLM

WHAT IF THE SYSTEM IS UNCERTAIN?
→ Human-in-the-loop

WHERE IS IT USED?
→ Web Dashboard
```

This notebook therefore completes the **Trust & Uncertainty layer** and prepares the project for the next stage: **RAG-based plant disease knowledge retrieval**.